# Harness de avaliacao de agente: trajetorias completas, nao so respostas

O notebook de comparacao de prompts (Fase 3) media **uma unica resposta** contra um gabarito. Um agente e mais dificil de avaliar porque produz uma **trajetoria**: uma sequencia de passos (chamadas de ferramenta, raciocinio intermediario) ate chegar (ou nao) numa resposta final. Dois agentes podem acertar a mesma tarefa por caminhos muito diferentes -- um em 2 passos, outro em 8, gastando 4x mais tokens.

Este notebook mede, para cada tarefa de um pequeno dataset:
- **Sucesso** (a resposta final passa num criterio objetivo?)
- **Numero de passos** ate concluir
- **Custo** (tokens de entrada + saida, servindo de proxy para custo em dinheiro)
- **Consistencia** (rodar a mesma tarefa varias vezes muda o resultado?)

**Pre-requisito**: harness do zero (projeto 1) -- reaproveitamos as ferramentas e a logica de loop de la.

**Sobre os testes**: toda a mecanica do harness de avaliacao e validada aqui com um cliente mock, sem custo de API real. A secao final mostra a mesma coisa com a API real.

## 1. Ferramentas reaproveitadas (projeto 1)

In [ ]:
import ast
import operator as op

_ALLOWED_OPS = {
    ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul, ast.Div: op.truediv,
    ast.Pow: op.pow, ast.USub: op.neg,
}

def _eval_node(node):
    if isinstance(node, ast.Constant):
        return node.value
    if isinstance(node, ast.BinOp):
        return _ALLOWED_OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp):
        return _ALLOWED_OPS[type(node.op)](_eval_node(node.operand))
    raise ValueError(f"Expressao nao suportada: {ast.dump(node)}")

def calculator(expression):
    try:
        return {"result": _eval_node(ast.parse(expression, mode="eval").body)}
    except Exception as e:
        return {"error": str(e)}


_FAKE_WEATHER_DB = {"sao paulo": {"temp_c": 22, "condicao": "nublado"}}

def get_weather(city):
    key = city.strip().lower()
    return _FAKE_WEATHER_DB.get(key, {"error": f"cidade \'{city}\' nao encontrada"})


TOOL_FUNCTIONS = {
    "calculator": lambda args: calculator(args["expression"]),
    "get_weather": lambda args: get_weather(args["city"]),
}

def execute_tool(name, args):
    if name not in TOOL_FUNCTIONS:
        return {"error": f"ferramenta desconhecida: {name}"}
    try:
        return TOOL_FUNCTIONS[name](args)
    except Exception as e:
        return {"error": str(e)}


## 2. O harness com rastreamento de trajetoria (`run_agent_traced`)

Diferenca em relacao ao `run_agent` do projeto 1: em vez de devolver so a resposta final, este harness registra **cada chamada de ferramenta** (nome, argumentos, resultado) e **os tokens consumidos em cada passo** -- e exatamente essa informacao extra que permite avaliar a trajetoria inteira, nao so o resultado.

In [ ]:
import json

def run_agent_traced(client, messages, max_steps=6):
    trajectory = {"tool_calls": [], "tokens_input": 0, "tokens_output": 0}
    steps = 0
    while steps < max_steps:
        steps += 1
        response = client.create(messages=messages)
        trajectory["tokens_input"] += response.usage.input_tokens
        trajectory["tokens_output"] += response.usage.output_tokens

        text_blocks = [b for b in response.content if b.type == "text"]
        tool_blocks = [b for b in response.content if b.type == "tool_use"]
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason != "tool_use":
            trajectory["status"] = "concluido"
            trajectory["resposta"] = " ".join(b.text for b in text_blocks)
            trajectory["passos"] = steps
            return trajectory

        tool_results = []
        for block in tool_blocks:
            result = execute_tool(block.name, block.input)
            trajectory["tool_calls"].append({
                "passo": steps, "ferramenta": block.name, "args": block.input, "resultado": result
            })
            tool_results.append({
                "type": "tool_result", "tool_use_id": block.id,
                "content": json.dumps(result, ensure_ascii=False),
            })
        messages.append({"role": "user", "content": tool_results})

    trajectory["status"] = "limite_de_passos"
    trajectory["resposta"] = None
    trajectory["passos"] = steps
    return trajectory


## 3. Dataset de avaliacao

Cada tarefa tem uma pergunta e uma funcao `verificar(resposta) -> bool` que define sucesso objetivamente -- o mesmo principio do harness de avaliacao de prompts da Fase 3, so que agora aplicado a resposta final de uma trajetoria inteira, nao a extracao de um campo estruturado.

In [ ]:
eval_dataset = [
    {
        "pergunta": "Quanto e 12 vezes 7?",
        "verificar": lambda resposta: resposta is not None and "84" in resposta,
    },
    {
        "pergunta": "Qual o clima em Sao Paulo?",
        "verificar": lambda resposta: resposta is not None and ("22" in resposta or "nublado" in resposta.lower()),
    },
    {
        "pergunta": "Qual o clima em Atlantis?",
        "verificar": lambda resposta: resposta is not None and ("nao encontr" in resposta.lower() or "desculp" in resposta.lower()),
    },
]


## 4. Testando o harness com um cliente mock

Simulamos as respostas do modelo para cada tarefa do dataset, incluindo contagem de tokens (`FakeUsage`), para validar que o rastreamento de trajetoria, sucesso e custo funcionam corretamente antes de gastar chamadas reais.

In [ ]:
class FakeToolUseBlock:
    def __init__(self, id, name, input_):
        self.type = "tool_use"; self.id = id; self.name = name; self.input = input_

class FakeTextBlock:
    def __init__(self, text):
        self.type = "text"; self.text = text

class FakeUsage:
    def __init__(self, input_tokens, output_tokens):
        self.input_tokens = input_tokens; self.output_tokens = output_tokens

class FakeResponse:
    def __init__(self, content, stop_reason, usage=None):
        self.content = content; self.stop_reason = stop_reason
        self.usage = usage or FakeUsage(50, 20)

class FakeAnthropicClient:
    def __init__(self, scripted_responses):
        self.scripted_responses = scripted_responses
        self.call_count = 0
    def create(self, **kwargs):
        r = self.scripted_responses[self.call_count]
        self.call_count += 1
        return r


scripts = [
    [FakeResponse([FakeToolUseBlock("t1", "calculator", {"expression": "12*7"})], "tool_use", FakeUsage(60, 15)),
     FakeResponse([FakeTextBlock("O resultado e 84.")], "end_turn", FakeUsage(70, 10))],

    [FakeResponse([FakeToolUseBlock("t1", "get_weather", {"city": "Sao Paulo"})], "tool_use", FakeUsage(55, 15)),
     FakeResponse([FakeTextBlock("Esta 22 graus e nublado em Sao Paulo.")], "end_turn", FakeUsage(65, 12))],

    [FakeResponse([FakeToolUseBlock("t1", "get_weather", {"city": "Atlantis"})], "tool_use", FakeUsage(55, 15)),
     FakeResponse([FakeTextBlock("Desculpe, nao encontrei essa cidade.")], "end_turn", FakeUsage(65, 12))],
]


## 5. Rodar o harness contra o dataset e agregar metricas

In [ ]:
import pandas as pd

results = []
for task, script in zip(eval_dataset, scripts):
    client = FakeAnthropicClient(script)
    traj = run_agent_traced(client, [{"role": "user", "content": task["pergunta"]}])
    sucesso = task["verificar"](traj["resposta"])
    custo_tokens = traj["tokens_input"] + traj["tokens_output"]
    results.append({
        "pergunta": task["pergunta"],
        "sucesso": sucesso,
        "passos": traj["passos"],
        "n_tool_calls": len(traj["tool_calls"]),
        "tokens_totais": custo_tokens,
    })

df = pd.DataFrame(results)
df


In [ ]:
taxa_sucesso = df["sucesso"].mean()
media_passos = df["passos"].mean()
custo_total = df["tokens_totais"].sum()

print(f"Taxa de sucesso: {taxa_sucesso:.0%}")
print(f"Media de passos por tarefa: {media_passos:.1f}")
print(f"Tokens totais (todas as tarefas): {custo_total}")


## 6. Teste de consistencia (mesma tarefa, execucoes repetidas)

Rodar a mesma tarefa varias vezes revela se o agente e confiavel ou se o comportamento varia entre execucoes. Aqui simulamos 3 execucoes da mesma pergunta, onde a terceira "erra" (o modelo passa um argumento diferente para a calculadora) -- para deixar claro que o harness detecta essa inconsistencia.

In [ ]:
task = eval_dataset[0]

execucoes = [
    [FakeResponse([FakeToolUseBlock("t1", "calculator", {"expression": "12*7"})], "tool_use"),
     FakeResponse([FakeTextBlock("O resultado e 84.")], "end_turn")],
    [FakeResponse([FakeToolUseBlock("t1", "calculator", {"expression": "12*7"})], "tool_use"),
     FakeResponse([FakeTextBlock("O resultado e 84.")], "end_turn")],
    [FakeResponse([FakeToolUseBlock("t1", "calculator", {"expression": "12*6"})], "tool_use"),  # modelo errou o input
     FakeResponse([FakeTextBlock("O resultado e 72.")], "end_turn")],
]

sucessos = []
for script in execucoes:
    client = FakeAnthropicClient(script)
    traj = run_agent_traced(client, [{"role": "user", "content": task["pergunta"]}])
    sucesso = task["verificar"](traj["resposta"])
    sucessos.append(sucesso)
    print(f"resposta: \'{traj['resposta']}\' | sucesso: {sucesso}")

taxa_consistencia = sum(sucessos) / len(sucessos)
print(f"\nTaxa de consistencia: {taxa_consistencia:.0%}")


## 7. Visualizar taxa de sucesso e custo por tarefa

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(range(len(df)), df["sucesso"].astype(int))
axes[0].set_xticks(range(len(df)))
axes[0].set_xticklabels([p[:15] + "..." for p in df["pergunta"]], rotation=30, ha="right")
axes[0].set_title("Sucesso por tarefa")
axes[0].set_ylim(0, 1.2)

axes[1].bar(range(len(df)), df["tokens_totais"], color="orange")
axes[1].set_xticks(range(len(df)))
axes[1].set_xticklabels([p[:15] + "..." for p in df["pergunta"]], rotation=30, ha="right")
axes[1].set_title("Tokens consumidos por tarefa")

plt.tight_layout()
plt.show()


## 8. Bonus: LLM como juiz da trajetoria

Alem de "a resposta final esta certa?", vale perguntar "o caminho ate a resposta foi razoavel?" -- um agente pode acertar a resposta por sorte, ou pode ter usado uma ferramenta de forma redundante varias vezes antes de acertar. Aqui usamos o proprio Claude para julgar a trajetoria completa (nao so a resposta final).

**Requer**: `pip install anthropic` e `ANTHROPIC_API_KEY`.

In [ ]:
import os
from anthropic import Anthropic

client_real = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

def julgar_trajetoria(pergunta, trajetoria):
    passos_desc = "\n".join(
        f"Passo {t['passo']}: chamou {t['ferramenta']}({t['args']}) -> {t['resultado']}"
        for t in trajetoria["tool_calls"]
    )
    prompt = f"""Avalie a trajetoria de um agente de IA que tentou responder a uma pergunta.

Pergunta: {pergunta}

Passos executados:
{passos_desc}

Resposta final: {trajetoria['resposta']}

Em uma escala de 1 a 5, quao eficiente e razoavel foi essa trajetoria (numero minimo de passos, ferramentas certas, sem redundancia)? Responda no formato: NOTA: X | JUSTIFICATIVA: ..."""

    response = client_real.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=200,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text


# exemplo de uso (requer ter uma trajetoria real de uma execucao com a API de verdade)
# print(julgar_trajetoria(task["pergunta"], traj))


## Notas finais

- **Diferenca central em relacao ao harness da Fase 3**: la, o "gabarito" era um valor exato esperado (nome, data, valor extraidos). Aqui, o "gabarito" e uma **funcao de verificacao** mais flexivel, porque a resposta de um agente varia mais em forma (ainda que o conteudo esteja certo) -- por isso `verificar()` busca por substrings/padroes em vez de igualdade exata
- **Metricas que importam em producao, alem de sucesso**: custo por tarefa (tokens), latencia (tempo de execucao, nao medido aqui mas facil de adicionar com `time.time()`), e taxa de recuperacao de erro (quantas vezes o agente encontrou um erro de ferramenta e ainda assim conseguiu completar a tarefa)
- **Dataset maior**: 3 tarefas so ilustra o mecanismo -- para conclusoes confiaveis sobre a qualidade de um agente, considere um dataset de pelo menos 20-30 tarefas cobrindo diferentes combinacoes de ferramentas e niveis de dificuldade
- **Conexao com o resto da fase**: este harness pode avaliar qualquer agente construido nos projetos anteriores desta fase -- harness manual, versao LangGraph, ou o agente com multiplas ferramentas e memoria -- desde que voce padronize a interface (`.create(messages=...)` devolvendo `.content`, `.stop_reason`, `.usage`)
